# Evaluating Agent Memory with MemoRizz

Agent memory is not one score. A trustworthy evaluation separates:

- **retrieval:** did the right evidence enter the context?
- **reader / agent:** did the system use that evidence correctly?
- **grounding:** can the answer identify valid sources?
- **lifecycle:** do updates, forgetting, summaries, cache, and scope behave correctly?
- **efficiency:** what latency, token, byte, and cost overhead did memory add?
- **protocol validity:** is this a smoke diagnostic, a regression run, or a paper-comparable result?

This notebook uses MemoRizz's public evaluation primitives and real filesystem provider. It is deliberately a bounded local diagnostic and makes no leaderboard claim.


## What you will learn

1. Inspect MemoRizz's benchmark catalog and evaluation profiles.
2. Build normalized `MemoryDocument` and `MemoryBenchmarkCase` records.
3. Derive source-linked semantic memories without reading gold labels.
4. Fuse semantic and lexical rankings with weighted reciprocal-rank fusion.
5. Compute Recall@k, MRR, nDCG, answer F1, exact match, citation, and grounding metrics.
6. Use a gold-evidence reader ceiling to localize retrieval versus reader failure.
7. Test real provider scope, update, delete/forget, provenance, and compaction links.
8. Apply MemoRizz's fail-closed paper-comparability gate.


## MemoRizz evaluation stack

| Layer | API used here | Evidence |
|---|---|---|
| Normalized cases | `MemoryDocument`, `MemoryBenchmarkCase` | stable source IDs, answers, rubrics, scorer family |
| Semantic derivation | `derive_semantic_memories` | constraint/preference/state records linked to their source |
| Retrieval | `build_query_variants`, `fuse_rankings` | label-blind lanes, fusion scores, final ranks |
| Retrieval scoring | `retrieval_metrics` | Recall@k, MRR, nDCG over source provenance |
| Answer scoring | `score_answer` | F1, exact, substring, Recall@5, or local judge contract |
| Provider lifecycle | `FileSystemProvider` | scoped durable reads/writes, update, forget, capability metadata |
| Protocol control | profiles + `assess_comparability` | diagnostic versus official/paper-comparable claim |

Full official datasets are intentionally external to the wheel. This notebook teaches the contracts with a small synthetic corpus; the MemoRizz CLI runs pinned official adapters when their datasets and graders are available.


## Setup and version preflight


In [1]:
from __future__ import annotations

import hashlib
import json
import math
import re
import tempfile
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

import memorizz
from memorizz import FileSystemConfig, FileSystemProvider, MemoryType
from memorizz.benchmarks.memory_suite import (
    BENCHMARK_CATALOG, EVALUATION_PROFILES, FusionConfig,
    MemoryBenchmarkCase, MemoryDocument, assess_comparability,
    build_query_variants, derive_semantic_memories, fuse_rankings,
    get_protocol_manifest,
)
from memorizz.benchmarks.memory_suite.scoring import (
    answer_f1, exact_match, retrieval_metrics, score_answer,
)

version = tuple(int(part) for part in memorizz.__version__.split(".")[:3])
if version < (0, 6, 3):
    raise RuntimeError("This notebook requires memorizz>=0.6.3,<0.7")
print("MemoRizz", memorizz.__version__, "· evaluation APIs ready")


MemoRizz 0.6.3 · evaluation APIs ready


## 1. Catalog, profiles, and claim boundaries


In [2]:
catalog_rows = []
for benchmark_id, spec in BENCHMARK_CATALOG.items():
    catalog_rows.append({
        "benchmark": benchmark_id,
        "description": getattr(spec, "description", ""),
        "variants": ", ".join(getattr(spec, "variants", ()) or ()),
    })
display(pd.DataFrame(catalog_rows).set_index("benchmark"))
display(pd.DataFrame([profile.to_dict() for profile in EVALUATION_PROFILES.values()]).set_index("name"))


,description,variants
benchmark,,
agentmembench,"Retrieval quality, answer quality, footprint, ...","locomo, multidoc2dial, msc"
longmemeval-v2,Compact evidence gathering over long web-agent...,"small-web, small-enterprise, medium-web, mediu..."
locomo-plus,Six-category conversational evaluation includi...,"cognitive, original, all"
beam,Long coherent conversations from 128K to 10M t...,"128k, 500k, 1m, 10m"
memoryagentbench,Incremental inject-once/query-many evaluation ...,"accurate-retrieval, test-time-learning, long-r..."


,description,default_limit,stratified
name,,,
smoke,"One bounded case per available category, for i...",NaN,True
regression,A fixed stratified subset suitable for repeata...,50.0,True
paper,The complete official split through the unmodi...,NaN,True


- **Smoke** checks dependencies and integration with one deterministic case per category.
- **Regression** uses a fixed stratified subset for repeatable A/B comparisons.
- **Paper** means the full split, but selecting it is not sufficient: the official runner, scorer, dataset revision, models, decoding, hardware, seed, and hashes must also match.

This distinction prevents a successful local harness run from being presented as a leaderboard result.


## 2. Build a source-linked memory corpus


In [3]:
documents = [
    MemoryDocument(
        source_id="turn-001",
        content="Ada owns the retrieval-api service and prefers Slack alerts.",
        metadata={"thread_id": "incident-7"},
    ),
    MemoryDocument(
        source_id="doc-rate",
        content="The Acme Cloud Pro plan permits 1,000 API requests per minute.",
        metadata={"title": "API Rate Limits"},
    ),
    MemoryDocument(
        source_id="turn-002",
        content="The retrieval queue reached 84 jobs on 24 August 2026, so Ada started draining old jobs.",
        metadata={
            "event_time": "2026-08-24T10:00:00+00:00",
            "semantic_group_id": "incident-84",
        },
    ),
    MemoryDocument(
        source_id="turn-003",
        content="Ada must not replace the live vector index during initial triage because it protects customer traffic.",
        metadata={"semantic_group_id": "incident-84"},
    ),
    MemoryDocument(
        source_id="doc-billing",
        content="Invoices are generated on the first day of each month.",
        metadata={"title": "Billing"},
    ),
]

derived = derive_semantic_memories(documents)
print(f"original={len(documents)} · derived semantic memories={len(derived)}")
display(pd.DataFrame([
    {
        "source_id": document.source_id,
        "parent_source_id": document.parent_source_id,
        "linked_source_ids": list(document.linked_source_ids),
        "type": document.metadata.get("semantic_memory_type"),
        "preview": document.content.replace("\n", " · ")[:130],
    }
    for document in derived
]))


original=5 · derived semantic memories=3


,source_id,parent_source_id,linked_source_ids,type,preview
0,turn-002#semantic-1,turn-002,[],state-update,Semantic memory type: state-update · Concepts:...
1,turn-003#semantic-1,turn-003,[],constraint,Semantic memory type: constraint · Source stat...
2,incident-84#semantic-summary,turn-002,"[turn-002, turn-003]",event-summary,Semantic memory type: event-summary · Concepts...


The derivation step is **query-independent**: it does not inspect the test question, relevant IDs, or reference answer. That matters because using labels during indexing leaks the benchmark answer into retrieval. Every derived item keeps parent or multi-source provenance so a hit can still receive source-level credit.


## 3. Label-blind query expansion and ranking fusion


In [4]:
def token_set(text):
    return set(re.findall(r"[a-z0-9_-]+", str(text).lower()))

def lexical_rank(query, rows):
    q = token_set(query)
    scored = []
    for row in rows:
        overlap = len(q & token_set(row.content))
        scored.append((overlap, row))
    scored.sort(key=lambda item: (-item[0], item[1].source_id))
    return [{**row.to_dict(), "score": float(score)} for score, row in scored]

class DeterministicEmbedder:
    dimensions = 64
    synonyms = {"overload":"queue", "backlog":"queue", "owner":"owns", "throttle":"limit"}
    def get_embedding(self, text):
        vector = np.zeros(self.dimensions, dtype=np.float32)
        for token in token_set(text):
            token = self.synonyms.get(token, token)
            digest = hashlib.sha256(token.encode()).digest()
            vector[int.from_bytes(digest[:4],"big") % self.dimensions] += 1 if digest[4] % 2 else -1
        norm = np.linalg.norm(vector)
        return (vector / norm if norm else vector).tolist()

embedder = DeterministicEmbedder()

def semantic_rank(query, rows):
    query_vector = np.asarray(embedder.get_embedding(query))
    scored = []
    for row in rows:
        score = float(query_vector @ np.asarray(embedder.get_embedding(row.content)))
        scored.append((score, row))
    scored.sort(key=lambda item: (-item[0], item[1].source_id))
    return [{**row.to_dict(), "score": score} for score, row in scored]

question = "What boundary protects Ada during retrieval overload?"
variants = build_query_variants(question, max_variants=3)
all_documents = [*documents, *derived]
lexical = lexical_rank(question, all_documents)
semantic_lanes = [(f"semantic:{i}", semantic_rank(variant, all_documents), 1.0)
                  for i, variant in enumerate(variants)]
fused = fuse_rankings(
    [*semantic_lanes, ("lexical", lexical, 0.35)],
    question=question,
    query_variants=variants,
    config=FusionConfig(top_k=4, candidate_pool_size=32),
    query_time="2026-08-25T09:00:00+00:00",
)
print("query variants:", variants)
display(pd.DataFrame([{
    "rank": row["_retrieval"]["final_rank"],
    "source_id": row["source_id"],
    "parent": row.get("parent_source_id"),
    "fusion": row["_retrieval"]["fusion_score"],
    "lanes": row["_retrieval"]["lane_ranks"],
} for row in fused]))


query variants: ['What boundary protects Ada during retrieval overload?', 'Retrieve prior memory about capacity, workload, commitments, overwhelm, stress, boundaries, and priorities; boundaries, declining obligations, saying no, protecting time, capacity, and reducing stress.']


,rank,source_id,parent,fusion,lanes
0,1,incident-84#semantic-summary,turn-002,1.097633,"{'semantic:0': 1, 'semantic:1': 2, 'lexical': 1}"
1,2,turn-001,turn-001,0.990249,"{'semantic:0': 4, 'semantic:1': 7, 'lexical': 6}"
2,3,turn-003#semantic-1,turn-003,1.031517,"{'semantic:0': 2, 'semantic:1': 4, 'lexical': 2}"
3,4,doc-rate,doc-rate,0.973312,"{'semantic:0': 8, 'semantic:1': 1, 'lexical': 8}"


## 4. Retrieval metrics


In [5]:
relevant_sources = ["turn-003"]
retrieved_metrics = retrieval_metrics(fused, relevant_sources)
display(pd.Series(retrieved_metrics, name="score").to_frame())


,score
recall_at_k,1.0
mrr,1.0
ndcg_at_k,1.0


- **Recall@k** credits a derived record when its `parent_source_id` or `linked_source_ids` contains a gold source. It measures coverage, not ranking.
- **MRR** is `1 / first relevant rank`; later relevant documents do not matter.
- **nDCG@k** discounts lower ranks and normalizes against an ideal ordering.

MemoRizz keeps these metrics separate from the downstream task score. A system can retrieve perfectly and still answer poorly; it can also guess a correct answer without retrieving its evidence.


## 5. Answer scoring and the reader ceiling


In [6]:
case = MemoryBenchmarkCase.create(
    case_id="acme-boundary-1", benchmark_id="course-diagnostic",
    corpus_id="acme-incident-84", category="constraint-recall",
    documents=documents, question=question,
    answers=["Do not replace the live vector index during initial triage."],
    relevant_source_ids=relevant_sources, scorer="answer_f1",
)
predictions = {
    "retrieved_evidence_reader": "Do not replace the live vector index during initial triage.",
    "verbose_but_correct": "Ada should not replace the live vector index during initial triage because customer traffic must remain protected.",
    "plausible_but_ungrounded": "Immediately rebuild the vector index to clear the queue.",
}
answer_rows = []
for lane, prediction in predictions.items():
    scored = score_answer(case, prediction)
    answer_rows.append({"lane":lane,"prediction":prediction,"metric":scored["metric"],
                        "score":scored["score"],"correct":scored["correct"]})
answer_df = pd.DataFrame(answer_rows).set_index("lane")
display(answer_df)


,prediction,metric,score,correct
lane,,,,
retrieved_evidence_reader,Do not replace the live vector index during in...,answer_f1,1.00,True
verbose_but_correct,Ada should not replace the live vector index d...,answer_f1,0.64,False
plausible_but_ungrounded,Immediately rebuild the vector index to clear ...,answer_f1,0.25,False


**Answer F1 is token overlap, not semantic truth.** A complete sentence may score lower than a terse reference because it contains extra tokens. Exact match is stricter still. Use deterministic scorers when they match the benchmark contract; add a calibrated judge only for semantics the deterministic scorer cannot express.

A **gold-evidence oracle reader** receives the known relevant sources. Its score estimates the reader ceiling:

- retrieved reader low, gold reader high → retrieval/context assembly problem;
- both low → reader, prompt, or scorer problem;
- retrieved score high without valid evidence → possible memorization or ungrounded guess.


In [7]:
lanes = pd.DataFrame([
    {"case":"A","retrieval_recall":0.0,"retrieved_reader":0.0,"gold_reader":1.0,"diagnosis":"retrieval miss"},
    {"case":"B","retrieval_recall":1.0,"retrieved_reader":0.5,"gold_reader":0.5,"diagnosis":"reader/scorer ceiling"},
    {"case":"C","retrieval_recall":1.0,"retrieved_reader":1.0,"gold_reader":1.0,"diagnosis":"end-to-end success"},
    {"case":"D","retrieval_recall":0.0,"retrieved_reader":1.0,"gold_reader":1.0,"diagnosis":"correct but ungrounded"},
]).set_index("case")
display(lanes)


,retrieval_recall,retrieved_reader,gold_reader,diagnosis
case,,,,
A,0.0,0.0,1.0,retrieval miss
B,1.0,0.5,0.5,reader/scorer ceiling
C,1.0,1.0,1.0,end-to-end success
D,0.0,1.0,1.0,correct but ungrounded


## 6. Citation and grounding metrics


In [8]:
def citation_metrics(cited_source_ids, supporting_source_ids, factual_claims, supported_claims):
    cited, supporting = set(cited_source_ids), set(supporting_source_ids)
    valid = cited & supporting
    return {
        "citation_precision": len(valid)/len(cited) if cited else 0.0,
        "citation_recall": len(valid)/len(supporting) if supporting else 0.0,
        "groundedness": supported_claims/factual_claims if factual_claims else 1.0,
        "citation_valid": float(cited <= supporting),
    }

citation_examples = [
    {"answer":"correct + cited","cited":["turn-003"],"support":["turn-003"],"claims":2,"supported":2},
    {"answer":"citation decoration","cited":["doc-billing"],"support":["turn-003"],"claims":2,"supported":1},
    {"answer":"supported but uncited","cited":[],"support":["turn-003"],"claims":2,"supported":2},
]
citation_df = pd.DataFrame([
    {"answer":row["answer"], **citation_metrics(row["cited"],row["support"],row["claims"],row["supported"])}
    for row in citation_examples
]).set_index("answer")
display(citation_df)


,citation_precision,citation_recall,groundedness,citation_valid
answer,,,,
correct + cited,1.0,1.0,1.0,1.0
citation decoration,0.0,0.0,0.5,0.0
supported but uncited,0.0,0.0,1.0,1.0


## 7. Real MemoRizz provider lifecycle evaluation


In [9]:
provider_root = Path(tempfile.mkdtemp(prefix="memorizz-eval-course-"))
provider = FileSystemProvider(FileSystemConfig(
    root_path=provider_root, use_faiss=False, embedding_provider=embedder
))
memory_id, user_id, thread_id = "acme-eval", "ada", "incident-84"
rows = [
    {"_id":"m1","content":"Ada owns retrieval-api.","memory_id":memory_id,"user_id":user_id,"thread_id":thread_id,"source_id":"turn-001"},
    {"_id":"m2","content":"The retrieval queue reached 84 jobs.","memory_id":memory_id,"user_id":user_id,"thread_id":thread_id,"source_id":"turn-002"},
    {"_id":"m3","content":"Do not replace the live vector index during triage.","memory_id":memory_id,"user_id":user_id,"thread_id":thread_id,"source_id":"turn-003"},
    {"_id":"m4","content":"Another tenant owns billing-api.","memory_id":memory_id,"user_id":"grace","thread_id":"billing-2","source_id":"other-001"},
]
started = time.perf_counter()
stored_ids = provider.store_many(rows, MemoryType.CONVERSATION_MEMORY, memory_id=memory_id)
ingest_ms = (time.perf_counter()-started)*1000

started = time.perf_counter()
scoped_hits = provider.search_memory(
    "What must Ada avoid during vector overload?", MemoryType.CONVERSATION_MEMORY,
    limit=3, memory_id=memory_id, user_id=user_id, thread_id=thread_id,
)
retrieval_ms = (time.perf_counter()-started)*1000
wrong_scope_hits = provider.search_memory(
    "billing owner", MemoryType.CONVERSATION_MEMORY,
    limit=3, memory_id=memory_id, user_id=user_id, thread_id=thread_id,
)
scope_violations = sum(row.get("user_id") != user_id or row.get("thread_id") != thread_id
                       for row in [*scoped_hits, *wrong_scope_hits])

# Update and forget tests.
provider.update_by_id("m1", {"content":"Ada now owns retrieval-api and cache-api."},
                      MemoryType.CONVERSATION_MEMORY)
update_correct = "cache-api" in provider.retrieve_by_id("m1", MemoryType.CONVERSATION_MEMORY)["content"]
provider.delete_by_id("m2", MemoryType.CONVERSATION_MEMORY)
forget_effective = provider.retrieve_by_id("m2", MemoryType.CONVERSATION_MEMORY) is None

lifecycle = pd.Series({
    "batch_write_success": len(stored_ids)/len(rows),
    "retrieval_recall@3": retrieval_metrics(scoped_hits, ["turn-003"])["recall_at_k"],
    "scope_isolation": float(scope_violations == 0),
    "update_correctness": float(update_correct),
    "forget_effectiveness": float(forget_effective),
    "provenance_coverage": np.mean([bool(row.get("source_id")) for row in scoped_hits]) if scoped_hits else 0.0,
    "ingest_ms": ingest_ms,
    "retrieval_ms": retrieval_ms,
}, name="value")
display(lifecycle.to_frame())
print("capabilities:", provider.memory_capabilities().to_dict())


,value
batch_write_success,1.000000
retrieval_recall@3,1.000000
scope_isolation,1.000000
update_correctness,1.000000
forget_effectiveness,1.000000
provenance_coverage,1.000000
ingest_ms,0.792209
retrieval_ms,0.232875


capabilities: {'provider': 'FileSystemProvider', 'batch_store': True, 'transactional_batch': False, 'scoped_search': True, 'result_scores': True, 'provenance': True, 'native_vector_search': False, 'native_hybrid_search': False}


Scope isolation is a hard gate: one cross-tenant row is not “99% good.” Update correctness should verify the new state and, in a temporal benchmark, ensure stale state no longer wins. Forgetting must test both direct retrieval and semantic indexes/caches; deleting only a source file while leaving an index entry is not effective forgetting.


## 8. Summary, compaction, and cache evidence


In [10]:
source_ids = ["m1","m3"]
summary_id = provider.store({
    "_id":"summary-1", "summary_id":"summary-1", "memory_id":memory_id,
    "user_id":user_id, "thread_id":thread_id,
    "content":"Ada owns retrieval-api and must keep the live vector index online during triage.",
    "source_message_ids":source_ids,
}, MemoryType.SUMMARIES, memory_id=memory_id)
for source_id in source_ids:
    provider.update_by_id(source_id, {"summary_id":summary_id}, MemoryType.CONVERSATION_MEMORY)

linked = [provider.retrieve_by_id(source_id, MemoryType.CONVERSATION_MEMORY) for source_id in source_ids]
compaction_link_coverage = np.mean([row.get("summary_id") == summary_id for row in linked])

cache_context = {"data_version":"acme-v1","scope":thread_id}
cache_key = hashlib.sha256(json.dumps({"q":"rate limit","context":cache_context},sort_keys=True).encode()).hexdigest()
provider.store({"_id":cache_key,"query":"rate limit","response":"1,000 requests/minute",
                "memory_id":memory_id,"user_id":user_id,"thread_id":thread_id,
                "context":cache_context}, MemoryType.SEMANTIC_CACHE, memory_id=memory_id)
cached = provider.retrieve_by_id(cache_key, MemoryType.SEMANTIC_CACHE)

consolidation = pd.Series({
    "summary_created":1.0,
    "compaction_link_coverage":compaction_link_coverage,
    "cache_exact_repeat_hit":float(cached is not None),
    "cache_scope_matches":float(cached["thread_id"] == thread_id),
    "cache_data_version_matches":float(cached["context"]["data_version"] == "acme-v1"),
}, name="score")
display(consolidation.to_frame())


,score
summary_created,1.0
compaction_link_coverage,1.0
cache_exact_repeat_hit,1.0
cache_scope_matches,1.0
cache_data_version_matches,1.0


A cache hit is not automatically correct. Record the tenant/thread scope, data version, age, admission reason, similarity, and whether side effects make reuse unsafe. Summary quality is also not just “a row exists”: measure source-link coverage, factual preservation, contradiction rate, compression ratio, and downstream retrieval/answer impact.


## 9. Fail-closed paper comparability


In [11]:
manifest = get_protocol_manifest("longmemeval-v2")
diagnostic_evidence = {
    "official_runner": False,
    "official_scorer": False,
    "dataset_verified": False,
    "num_samples": 1,
    "reader_model": "deterministic-course-reader",
    "embedding_model": "hash-64",
    "judge_model": None,
    "top_k": 4,
    "dataset_revision": "synthetic-v1",
    "dataset_fingerprint": "sha256:course-fixture",
    "prompt_hash": "sha256:course-prompt",
    "scorer_hash": "sha256:memorizz-scoring",
    "dependency_lock_hash": "sha256:course-env",
    "hardware": "local workshop machine",
    "seed": 42,
    "upstream_revision": None,
}
comparability = assess_comparability(manifest, diagnostic_evidence, profile="smoke")
print("comparison label:", comparability["comparison_label"])
print("paper comparable:", comparability["paper_comparable"])
display(pd.DataFrame({"reason":comparability["non_comparability_reasons"]}))


comparison label: Diagnostic
paper comparable: False


,reason
0,"profile='smoke', not the full paper profile"
1,MemoRizz diagnostic runner used instead of the...
2,MemoRizz diagnostic scorer used instead of the...
3,official dataset assets were not verified
4,sample count 1 does not match the paper count 451
5,upstream source revision does not match the pi...
6,reader_model='deterministic-course-reader' doe...
7,embedding_model='hash-64' does not match 'Qwen...
8,judge_model=None does not match 'GPT-5.2'


## 10. Read a MemoRizz-style diagnostic report


In [12]:
report = {
    "paper_comparable": False,
    "comparison_label": "Diagnostic",
    "lanes": [
        {"lane":"retrieval path","task_score":1.0,"recall_at_k":1.0,"mrr":1.0,"gold_reader":1.0,"tokens":1619,"cost_usd":0.0},
        {"lane":"reader-limited path","task_score":0.0,"recall_at_k":1.0,"mrr":0.25,"gold_reader":0.5,"tokens":1652,"cost_usd":0.0},
        {"lane":"full MemAgent","task_score":0.5,"recall_at_k":1.0,"mrr":0.20,"gold_reader":None,"tokens":2438,"cost_usd":0.0},
    ],
}
report_df = pd.DataFrame(report["lanes"]).set_index("lane")
display(report_df)
print("The second lane is a reader/evidence-utilization problem, not a retrieval miss.")


,task_score,recall_at_k,mrr,gold_reader,tokens,cost_usd
lane,,,,,,
retrieval path,1.0,1.0,1.00,1.0,1619,0.0
reader-limited path,0.0,1.0,0.25,0.5,1652,0.0
full MemAgent,0.5,1.0,0.20,NaN,2438,0.0


The second lane is a reader/evidence-utilization problem, not a retrieval miss.


## A production memory evaluation matrix

| Capability | Primary metric | Required evidence |
|---|---|---|
| Recall | Recall@k, MRR, nDCG | ranked IDs, parent/linked source IDs, scores |
| Answer use | task score + gold-reader ceiling | prediction, references, retrieved and gold context |
| Grounding | citation precision/recall, supported-claim rate | claim-to-source links |
| Scope | isolation violations | memory/user/thread IDs on request and result |
| Update/conflict | latest-state accuracy, stale-state rate | version/timestamp and conflict lifecycle |
| Forgetting | deleted-item retrieval rate | source, index, summary, and cache deletion evidence |
| Summarization | link coverage, preservation, contradiction, compression | summary IDs and source-message IDs |
| Semantic cache | hit rate, precision, stale-hit rate, bypass rate | query/context hash, scope, version, age, decision |
| Tool/workflow memory | trajectory reuse and outcome delta | tool-log/workflow IDs and verified outcomes |
| Efficiency | p50/p95, tokens, bytes, cost per correct answer | stage timings and provider/model usage |
| Reproducibility | comparability gate | dataset/model/prompt/scorer revisions, seed, hardware |

Always pair a memory feature metric with an end-task metric. A smaller context is not an improvement if answer quality falls; a cache hit is not useful if it is stale; a summary is not successful if it drops the constraint needed later.


In [13]:
provider.close()
import shutil
shutil.rmtree(provider_root)
print("Temporary MemoRizz provider workspace removed.")


Temporary MemoRizz provider workspace removed.


## Where to go next

Use the package CLI for real benchmark adapters:

```bash
memorizz eval list
memorizz eval protocol show longmemeval-v2
memorizz eval dataset verify longmemeval-v2 --data-path /data/longmemeval-v2
memorizz eval run locomo-plus --data-path /data/Locomo-Plus/data --profile smoke
```

For an end-to-end agent-path diagnostic, select `evaluation_mode="memagent"` with a secret-free agent template. For a retrieval-focused diagnosis, use `evaluation_mode="retrieval"`. Keep the dataset, model, embeddings, seed, resources, and scorer fixed before interpreting a provider A/B comparison.
